## Schema Population

**Information Need:** Understand how a selected attribute is populated across cases.

**Motivation:** Attribute information may differ not only in its availability across activity types but also across cases. An attribute may be available exactly once per case (i.e., case attribute candidate), multiple times with the same or different values, or only for a subset of cases. Understanding these filling characteristics helps analysts assess the attribute's availability.

**Approach:** Derive the population of a selected attribute across cases based on its presence, frequency of occurrence, and consistency of values within each case.

**Output:** The distribution of the attribute's filling characteristics across cases, including the percentage of cases in which the attribute is missing, populated once, populated multiple times with the same value, or populated multiple times with different values.

In [ ]:
import pandas as pd
import pm4py

from ipywidgets import interact

# --- Configuration -----------------------------------------------------------
LOG_PATH = "../../data/Road_Traffic_Fine_Management_Process.xes"

CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
TIMESTAMP = "time:timestamp"

In [ ]:
event_log = pm4py.read_xes(LOG_PATH)

display(event_log.head())

## Pattern execution

In [ ]:
FILLING_CLASSES = ['missing', 'populated once', 'populated multiple times (same value)', 'populated multiple times (different values)']


def classify_case(values):
    non_null = values.dropna()
    if len(non_null) == 0:
        return 'missing'
    if len(non_null) == 1:
        return 'populated once'
    if non_null.nunique() == 1:
        return 'populated multiple times (same value)'
    return 'populated multiple times (different values)'


attribute_columns = [c for c in event_log.columns if c not in (CASE_ID, ACTIVITY, TIMESTAMP)]

In [ ]:
@interact(attribute=list(attribute_columns))
def compute_case_wise_filling(attribute):
    case_classification = event_log.groupby(CASE_ID)[attribute].apply(classify_case)

    counts = case_classification.value_counts().reindex(FILLING_CLASSES, fill_value=0)
    percentages = (counts / len(case_classification) * 100).round(2)

    distribution = pd.DataFrame({
        'Filling class': FILLING_CLASSES,
        'Cases': counts.values,
        'Percentage': percentages.values
    })

    display(distribution)